<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/egovlp_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EgoVLP Baseline

Trains and evaluates the **existing, unmodified** `MLP` and `Transformer` (`ErFormer`) architectures
on **EgoVLP** features (step split), using the same repo code already used for Omnivore --
`fetch_model` now also supports `backbone="egovlp"` (see `constants.py` / `core/models/blocks.py` /
`base.py` -- four small additive whitelist edits, no architecture changes). This isolates the effect
of the feature backbone: only the input features change, the classifiers are the same code.

Run `egovlp_feature_extraction.ipynb` first -- this notebook expects the extracted `.npz` files
already in `data/video/egovlp/` on Drive.

This is a separate notebook from the baseline reproduction, error-category-analysis, and LSTM
notebooks and does not modify any of them.

<a href="https://colab.research.google.com/github/zeynepoztunc/aml-procedural-mistake-detection/blob/zeynep-september/notebooks/egovlp_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Setup (Colab)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!rm -rf /content/code

In [4]:
!git clone --recursive --branch zeynep-september https://github.com/zeynepoztunc/aml-procedural-mistake-detection.git \
/content/code

Cloning into '/content/code'...
remote: Enumerating objects: 923, done.
remote: Counting objects: 100% (137/137), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 923 (delta 109), reused 95 (delta 91), pack-reused 786 (from 2)
Receiving objects: 100% (923/923), 96.55 MiB | 19.84 MiB/s, done.
Resolving deltas: 100% (499/499), done.
Submodule 'annotations' (https://github.com/CaptainCook4D/annotations) registered for path 'annotations'
Cloning into '/content/code/annotations'...
remote: Enumerating objects: 152, done.        
remote: Counting objects: 100% (152/152), done.        
remote: Compressing objects: 100% (98/98), done.        
remote: Total 152 (delta 75), reused 108 (delta 46), pack-reused 0 (from 0)        
Receiving objects: 100% (152/152), 793.14 KiB | 17.24 MiB/s, done.
Resolving deltas: 100% (75/75), done.
Submodule path 'annotations': checked out '0e9a108be2cbcbcbd592e7418c0ab9c16232d27a'


In [5]:
%cd /content/code

/content/code


In [6]:
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> select a GPU."
)

DEVICE = "cuda"

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Torch: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [7]:
!pip install -q torcheval pyrebase4 yacs loguru wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 6.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.2.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


In [8]:
!ls data/video/egovlp | head -10

ls: cannot access 'data/video/egovlp': No such file or directory


## 2. EgoVLP features

Copies the `.npz` files produced by `egovlp_feature_extraction.ipynb` (saved there directly as
`<recording_id>_360p.mp4_1s_1s.npz`, key `arr_0`) into `data/video/egovlp/` -- the same layout
`CaptainCookStepDataset` already uses for Omnivore, so no renaming or dataloader changes are
needed here.

In [9]:
import os
import numpy as np

DRIVE_BASE_PATH = "/content/drive/MyDrive/AML_Project"
DRIVE_EGOVLP_DIR = f"{DRIVE_BASE_PATH}/features/egovlp"

CODE_DIR = "/content/code"
EGOVLP_DIR = f"{CODE_DIR}/data/video/egovlp"

assert os.path.exists(DRIVE_EGOVLP_DIR), (
    f"Not found: {DRIVE_EGOVLP_DIR} -- run egovlp_feature_extraction.ipynb first."
)

os.makedirs(EGOVLP_DIR, exist_ok=True)

KNOWN_SUFFIXES = ["_360p_224.npz", "_360p.mp4_1s_1s.npz", ".npz"]

src_files = [f for f in os.listdir(DRIVE_EGOVLP_DIR) if f.endswith(".npz")]
print(f"Found {len(src_files)} source files in {DRIVE_EGOVLP_DIR}")

copied, skipped = 0, 0
for fname in src_files:
    recording_id = fname
    for suffix in KNOWN_SUFFIXES:
        if fname.endswith(suffix):
            recording_id = fname[: -len(suffix)]
            break

    dest_path = os.path.join(EGOVLP_DIR, f"{recording_id}_360p.mp4_1s_1s.npz")
    if os.path.exists(dest_path):
        skipped += 1
        continue

    data = np.load(os.path.join(DRIVE_EGOVLP_DIR, fname))
    if "arr_0" in data:
        features = data["arr_0"]
    elif "video_features" in data:
        features = data["video_features"]
    else:
        features = data[list(data.keys())[0]]

    np.savez(dest_path, arr_0=features)
    copied += 1

print(f"Copied {copied} files, skipped {skipped} already-present files.")

files = os.listdir(EGOVLP_DIR)
print(f"EgoVLP features ready: {len(files)} files")
print(f"Sample: {files[:3]}")



Found 384 source files in /content/drive/MyDrive/AML_Project/features/egovlp
Copied 384 files, skipped 0 already-present files.
EgoVLP features ready: 384 files
Sample: ['29_5_360p.mp4_1s_1s.npz', '28_26_360p.mp4_1s_1s.npz', '29_29_360p.mp4_1s_1s.npz']


## 3. Train + evaluate (val selects, test reported once)

Same building blocks and workflow as the other notebooks in this repo -- `fetch_model` (now
EgoVLP-aware, unmodified `MLP`/`ErFormer` classes), `CaptainCookStepDataset`/`collate_fn`,
`train_epoch`/`test_er_model` -- with no test leakage:

1. Train on `train`.
2. After every epoch, evaluate on `val` and record val AUC.
3. Keep the model weights from the epoch with the best val AUC (in memory).
4. Restore those weights after training finishes.
5. Evaluate **once** on `test`.

Same hyperparameters as the LSTM/error-category notebooks (lr=1e-3, weight_decay=1e-3,
pos_weight=2.5, batch_size=8, threshold=0.6) -- deliberately *not* the stability tweaks (Xavier
init, tiny LR, feature normalization) from the old reference notebook, since those change training
dynamics and would confound the backbone comparison. If this hits the existing NaN-loss assert in
`train_epoch`, that is itself a real finding about raw EgoVLP feature scale, not something to
silently patch around here.

In [10]:
sys.path.insert(0, ".")

from types import SimpleNamespace
from constants import Constants as const
from dataloader.CaptainCookStepDataset import CaptainCookStepDataset, collate_fn
from base import fetch_model, train_epoch, test_er_model
from torch.utils.data import DataLoader
import torch.nn as nn

BACKBONE = const.EGOVLP
SPLIT = const.STEP_SPLIT
TASK_NAME = const.ERROR_RECOGNITION
NUM_EPOCHS = 15
BATCH_SIZE = 8
THRESHOLD = 0.6


def run_egovlp_experiment(variant, num_epochs=NUM_EPOCHS, threshold=THRESHOLD):
    config = SimpleNamespace(
        backbone=BACKBONE,
        modality="video",
        segment_features_directory="data/",
        split=SPLIT,
        task_name=TASK_NAME,
        error_category=None,
        variant=variant,
        seed=1000,
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    torch.manual_seed(config.seed)

    train_dataset = CaptainCookStepDataset(config, const.TRAIN, config.split)
    val_dataset = CaptainCookStepDataset(config, const.VAL, config.split)
    test_dataset = CaptainCookStepDataset(config, const.TEST, config.split)

    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

    model = fetch_model(config)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.5], device=DEVICE))

    best_val_auc = -1.0
    best_epoch = None
    best_state_dict = None

    for epoch in range(1, num_epochs + 1):
        train_epoch(model, DEVICE, train_loader, optimizer, epoch, criterion)

        _, _, val_metrics = test_er_model(
            model, val_loader, criterion, DEVICE, phase="val", threshold=threshold
        )
        val_auc = float(val_metrics["auc"])
        print(f"  epoch {epoch}: val AUC = {val_auc:.4f}")

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_epoch = epoch
            best_state_dict = {k: v.detach().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state_dict)
    _, _, test_metrics = test_er_model(
        model, test_loader, criterion, DEVICE, phase="test", threshold=threshold
    )

    print(f"  selected best epoch = {best_epoch} (val AUC = {best_val_auc:.4f})")
    print(f"  final TEST metrics: {test_metrics}")

    return {
        "best_epoch": best_epoch,
        "val_auc": best_val_auc,
        "accuracy": test_metrics["accuracy"],
        "precision": test_metrics["precision"],
        "recall": test_metrics["recall"],
        "f1": test_metrics["f1"],
        "auc": test_metrics["auc"],
    }

### Smoke test

A couple of quick epochs on MLP + EgoVLP to confirm the pipeline runs end-to-end (features load,
`fetch_model` accepts `backbone="egovlp"`, no NaNs) before the full runs.

In [11]:
_smoke_metrics = run_egovlp_experiment(const.MLP_VARIANT, num_epochs=3)
print("\nReturned dict:")
print(_smoke_metrics)

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 468/469, Loss: 0.653907: 100%|██████████| 469/469 [00:08<00:00, 52.67it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 306.76it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.4989697802197802, 'recall': 0.10937147158449378, 'f1': 0.1794159412236834, 'accuracy': 0.613273975791434, 'auc': np.float64(0.5925929188132194), 'pr_auc': tensor(0.3988)}
val Step Level Metrics: {'precision': 0.39349112426035504, 'recall': 0.540650406504065, 'f1': 0.4554794520547945, 'accuracy': 0.5891472868217055, 'auc': np.float64(0.5891460335057896), 'pr_auc': tensor(0.3587)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.5891


Train Epoch: 2, Progress: 468/469, Loss: 0.398796: 100%|██████████| 469/469 [00:07<00:00, 65.29it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 290.92it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.484158720393725, 'recall': 0.1184794881445239, 'f1': 0.1903725205611998, 'accuracy': 0.610451582867784, 'auc': np.float64(0.5849562470006041), 'pr_auc': tensor(0.3981)}
val Step Level Metrics: {'precision': 0.37597911227154046, 'recall': 0.5853658536585366, 'f1': 0.4578696343402226, 'accuracy': 0.5594315245478036, 'auc': np.float64(0.5821938901207193), 'pr_auc': tensor(0.3519)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5822


Train Epoch: 3, Progress: 468/469, Loss: 1.339791: 100%|██████████| 469/469 [00:06<00:00, 67.93it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 314.54it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.48268398268398266, 'recall': 0.16785848701543093, 'f1': 0.24909243228148562, 'accuracy': 0.6087930633147114, 'auc': np.float64(0.5940457979127998), 'pr_auc': tensor(0.4027)}
val Step Level Metrics: {'precision': 0.38549618320610685, 'recall': 0.4105691056910569, 'f1': 0.39763779527559057, 'accuracy': 0.6046511627906976, 'auc': np.float64(0.5988159029317566), 'pr_auc': tensor(0.3456)}
----------------------------------------------------------------
  epoch 3: val AUC = 0.5988


test Progress: 42346/798: 100%|██████████| 798/798 [00:02<00:00, 319.81it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.46956329951477727, 'recall': 0.17960182216973172, 'f1': 0.25982426165486944, 'accuracy': 0.7135502762952817, 'auc': np.float64(0.6611811815166647), 'pr_auc': tensor(0.3140)}
test Step Level Metrics: {'precision': 0.4666666666666667, 'recall': 0.3373493975903614, 'f1': 0.3916083916083916, 'accuracy': 0.6729323308270677, 'auc': np.float64(0.6740916306391322), 'pr_auc': tensor(0.3642)}
----------------------------------------------------------------
  selected best epoch = 3 (val AUC = 0.5988)
  final TEST metrics: {'precision': 0.4666666666666667, 'recall': 0.3373493975903614, 'f1': 0.3916083916083916, 'accuracy': 0.6729323308270677, 'auc': np.float64(0.6740916306391322), 'pr_auc': tensor(0.3642)}

Returned dict:
{'best_epoch': 3, 'val_auc': 0.5988159029317566, 'accuracy': 0.6729323308270677, 'precision': 0.4666666666666667, 'recall': 0.3373493975903614, 'f1': 0.3916083916083916,

### Full runs

In [12]:
mlp_metrics = run_egovlp_experiment(const.MLP_VARIANT)
mlp_metrics

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 468/469, Loss: 0.653907: 100%|██████████| 469/469 [00:07<00:00, 59.87it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 300.94it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.4989697802197802, 'recall': 0.10937147158449378, 'f1': 0.1794159412236834, 'accuracy': 0.613273975791434, 'auc': np.float64(0.5925929188132194), 'pr_auc': tensor(0.3988)}
val Step Level Metrics: {'precision': 0.39349112426035504, 'recall': 0.540650406504065, 'f1': 0.4554794520547945, 'accuracy': 0.5891472868217055, 'auc': np.float64(0.5891460335057896), 'pr_auc': tensor(0.3587)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.5891


Train Epoch: 2, Progress: 468/469, Loss: 0.398796: 100%|██████████| 469/469 [00:06<00:00, 68.10it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 305.04it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.484158720393725, 'recall': 0.1184794881445239, 'f1': 0.1903725205611998, 'accuracy': 0.610451582867784, 'auc': np.float64(0.5849562470006041), 'pr_auc': tensor(0.3981)}
val Step Level Metrics: {'precision': 0.37597911227154046, 'recall': 0.5853658536585366, 'f1': 0.4578696343402226, 'accuracy': 0.5594315245478036, 'auc': np.float64(0.5821938901207193), 'pr_auc': tensor(0.3519)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5822


...

Train Epoch: 15, Progress: 468/469, Loss: 1.114136: 100%|██████████| 469/469 [00:06<00:00, 74.75it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 319.39it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.4590764892492069, 'recall': 0.49017689123071134, 'f1': 0.47411721878412816, 'accuracy': 0.5796671322160148, 'auc': np.float64(0.6000535189599842), 'pr_auc': tensor(0.4221)}
val Step Level Metrics: {'precision': 0.3969465648854962, 'recall': 0.42276422764227645, 'f1': 0.4094488188976378, 'accuracy': 0.6124031007751938, 'auc': np.float64(0.6215970682434097), 'pr_auc': tensor(0.3513)}
----------------------------------------------------------------
  epoch 15: val AUC = 0.6216


test Progress: 42346/798: 100%|██████████| 798/798 [00:02<00:00, 319.51it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.4155093590960062, 'recall': 0.4063607221191159, 'f1': 0.4108841216360302, 'accuracy': 0.6738062626930524, 'auc': np.float64(0.6604045960515583), 'pr_auc': tensor(0.3350)}
test Step Level Metrics: {'precision': 0.55, 'recall': 0.22088353413654618, 'f1': 0.3151862464183381, 'accuracy': 0.7005012531328321, 'auc': np.float64(0.7323282199837602), 'pr_auc': tensor(0.3646)}
----------------------------------------------------------------
  selected best epoch = 7 (val AUC = 0.6394)
  final TEST metrics: {'precision': 0.55, 'recall': 0.22088353413654618, 'f1': 0.3151862464183381, 'accuracy': 0.7005012531328321, 'auc': np.float64(0.7323282199837602), 'pr_auc': tensor(0.3646)}


{'best_epoch': 7,
 'val_auc': 0.6393662232076865,
 'accuracy': 0.7005012531328321,
 'precision': 0.55,
 'recall': 0.22088353413654618,
 'f1': 0.3151862464183381,
 'auc': np.float64(0.7323282199837602)}

In [13]:
transformer_metrics = run_egovlp_experiment(const.TRANSFORMER_VARIANT)
transformer_metrics

Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json
Loaded annotations...... 
Loading recording ids from recordings_combined_splits.json


Train Epoch: 1, Progress: 468/469, Loss: 0.784651: 100%|██████████| 469/469 [00:08<00:00, 54.34it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:02<00:00, 261.28it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.5212121212121212, 'recall': 0.09062852841550621, 'f1': 0.15440846425136262, 'accuracy': 0.6163000465549349, 'auc': np.float64(0.6019527618952003), 'pr_auc': tensor(0.3988)}
val Step Level Metrics: {'precision': 0.3688212927756654, 'recall': 0.7886178861788617, 'f1': 0.5025906735751295, 'accuracy': 0.5038759689922481, 'auc': np.float64(0.612389135254989), 'pr_auc': tensor(0.3580)}
----------------------------------------------------------------
  epoch 1: val AUC = 0.6124


Train Epoch: 2, Progress: 468/469, Loss: 1.112091: 100%|██████████| 469/469 [00:08<00:00, 55.52it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:03<00:00, 249.89it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.4827089337175792, 'recall': 0.05043281896876176, 'f1': 0.091324200913242, 'accuracy': 0.6120519087523277, 'auc': np.float64(0.5860482872763528), 'pr_auc': tensor(0.3914)}
val Step Level Metrics: {'precision': 0.3684210526315789, 'recall': 0.4268292682926829, 'f1': 0.3954802259887006, 'accuracy': 0.5852713178294574, 'auc': np.float64(0.5851964769647697), 'pr_auc': tensor(0.3394)}
----------------------------------------------------------------
  epoch 2: val AUC = 0.5852


...

Train Epoch: 15, Progress: 468/469, Loss: 1.583629: 100%|██████████| 469/469 [00:08<00:00, 53.72it/s]
val Progress: 34368/774: 100%|██████████| 774/774 [00:03<00:00, 245.36it/s]


----------------------------------------------------------------
val Sub Step Level Metrics: {'precision': 0.42946768060836504, 'recall': 0.6801656003010914, 'f1': 0.5264967225054625, 'accuracy': 0.5270891527001862, 'auc': np.float64(0.5741447940645581), 'pr_auc': tensor(0.4157)}
val Step Level Metrics: {'precision': 0.364, 'recall': 0.3699186991869919, 'f1': 0.36693548387096775, 'accuracy': 0.5943152454780362, 'auc': np.float64(0.5659645232815964), 'pr_auc': tensor(0.3349)}
----------------------------------------------------------------
  epoch 15: val AUC = 0.5660


test Progress: 42346/798: 100%|██████████| 798/798 [00:03<00:00, 238.85it/s]


----------------------------------------------------------------
test Sub Step Level Metrics: {'precision': 0.32677016281711474, 'recall': 0.07280242955964232, 'f1': 0.11907554329078993, 'accuracy': 0.6984603032163604, 'auc': np.float64(0.5431918089919993), 'pr_auc': tensor(0.2833)}
test Step Level Metrics: {'precision': 0.34791252485089463, 'recall': 0.7028112449799196, 'f1': 0.4654255319148936, 'accuracy': 0.49624060150375937, 'auc': np.float64(0.5678744120379514), 'pr_auc': tensor(0.3372)}
----------------------------------------------------------------
  selected best epoch = 1 (val AUC = 0.6124)
  final TEST metrics: {'precision': 0.34791252485089463, 'recall': 0.7028112449799196, 'f1': 0.4654255319148936, 'accuracy': 0.49624060150375937, 'auc': np.float64(0.5678744120379514), 'pr_auc': tensor(0.3372)}


{'best_epoch': 1,
 'val_auc': 0.612389135254989,
 'accuracy': 0.49624060150375937,
 'precision': 0.34791252485089463,
 'recall': 0.7028112449799196,
 'f1': 0.4654255319148936,
 'auc': np.float64(0.5678744120379514)}

## 4. Comparison against the Omnivore baseline

The Omnivore rows are the already-reproduced numbers from evaluating the CaptainCook4D
**pretrained checkpoints** (not retrained here) -- a different training procedure than the
val-selected EgoVLP runs above. This isn't a fully controlled A/B because of that, but it's the
same reference point already used in the other notebooks in this repo; noted here so the numbers
aren't over-interpreted.

In [14]:
import pandas as pd

OMNIVORE_BASELINES = [
    {"Model": "MLP", "Backbone": "Omnivore", "Accuracy": 71.05, "Precision": 66.07, "Recall": 14.86, "F1": 24.26, "AUC": 75.74},
    {"Model": "Transformer", "Backbone": "Omnivore", "Accuracy": 69.92, "Precision": 51.56, "Recall": 59.84, "F1": 55.39, "AUC": 75.62},
]

egovlp_rows = [
    {
        "Model": "MLP", "Backbone": "EgoVLP",
        "Accuracy": round(float(mlp_metrics["accuracy"]) * 100, 2),
        "Precision": round(float(mlp_metrics["precision"]) * 100, 2),
        "Recall": round(float(mlp_metrics["recall"]) * 100, 2),
        "F1": round(float(mlp_metrics["f1"]) * 100, 2),
        "AUC": round(float(mlp_metrics["auc"]) * 100, 2),
    },
    {
        "Model": "Transformer", "Backbone": "EgoVLP",
        "Accuracy": round(float(transformer_metrics["accuracy"]) * 100, 2),
        "Precision": round(float(transformer_metrics["precision"]) * 100, 2),
        "Recall": round(float(transformer_metrics["recall"]) * 100, 2),
        "F1": round(float(transformer_metrics["f1"]) * 100, 2),
        "AUC": round(float(transformer_metrics["auc"]) * 100, 2),
    },
]

comparison_df = pd.DataFrame(OMNIVORE_BASELINES + egovlp_rows)
comparison_df = comparison_df[["Model", "Backbone", "Accuracy", "Precision", "Recall", "F1", "AUC"]]
comparison_df = comparison_df.sort_values(["Model", "Backbone"]).reset_index(drop=True)
comparison_df

,Model,Backbone,Accuracy,Precision,Recall,F1,AUC
0,MLP,EgoVLP,70.05,55.00,22.09,31.52,73.23
1,MLP,Omnivore,71.05,66.07,14.86,24.26,75.74
2,Transformer,EgoVLP,49.62,34.79,70.28,46.54,56.79
3,Transformer,Omnivore,69.92,51.56,59.84,55.39,75.62


##Results
Replacing Omnivore with EgoVLP affected the two classifiers differently. For the MLP, EgoVLP improved F1 from 24.26% to 31.52%, mainly due to higher recall, while AUC slightly decreased from 75.74% to 73.23%. For the Transformer, EgoVLP substantially increased recall but reduced precision, accuracy, F1 and AUC compared with Omnivore. These results indicate that the EgoVLP representation does not consistently outperform Omnivore for supervised error recognition in this configuration.